## Middleware

-->Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

--> Tracking agent behavior with logging, analytics, and debugging.

--> Transforming prompts, tool selection, and output formatting.

--> Adding retries, fallbacks, and early termination logic.

--> Applying rate limits, guardrails, and PII detection.

In [10]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY") #type:ignore
os.environ["HUGGINGFACE_API_KEY"] = os.getenv("HUGGINGFACE_API_KEY") #type:ignore

## Summarization MiddleWare

 -->Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

--> Long-running conversations that exceed context windows.

--> Multi-turn dialogues with extensive history.

--> Applications where preserving full conversation context matters.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,AIMessage

### Messagebased summarization

agent = create_agent(
    model = "groq:openai/gpt-oss-120b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-120b",
            trigger = ("messages",10),
            keep = ("messages",4)
            
        )
    ]
    
)

In [21]:
### run with thread id 

config = {"configurable":{"thread_id":"test-1"}}

In [23]:
## alternative test data

questions = [
    "what is 2+2 ? ",
    "what is 10*5 ? ",
    "what is 100/4? ",
    "what is 15-7? ",
    "what is 3*3? ",
    "what is 4*4 ? "
]

for q in questions:
    response =  agent.invoke({"messages":[HumanMessage(content=q)]},config) #type:ignore
    print(f"messages : {response}")
    print(f"messages : {len(response['messages'])}")

messages : {'messages': [HumanMessage(content='what is 2+2 ? what is 10*5 ? what is 100/4? what is 15-7? what is 3*3? what is 4*4 ? ', additional_kwargs={}, response_metadata={}, id='b53853a7-8491-4789-8943-e1622ecf29a4'), AIMessage(content='Here are the results:\n\n- **2\u202f+\u202f2 = 4**  \n- **10\u202f×\u202f5 = 50**  \n- **100\u202f÷\u202f4 = 25**  \n- **15\u202f−\u202f7 = 8**  \n- **3\u202f×\u202f3 = 9**  \n- **4\u202f×\u202f4 = 16**', additional_kwargs={'reasoning_content': 'User asks simple arithmetic. Provide answers. Probably list them.'}, response_metadata={'token_usage': {'completion_tokens': 98, 'prompt_tokens': 114, 'total_tokens': 212, 'completion_time': 0.20321792, 'completion_tokens_details': {'reasoning_tokens': 13}, 'prompt_time': 0.004252671, 'prompt_tokens_details': None, 'queue_time': 0.401661099, 'total_time': 0.207470591}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_19b184c447', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs'

## token size 



In [26]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool 
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool 
def search_hostels(city:str) -> str:
    """
    Search hostels - return long response to use more tokens
    
    """
    return f""" hotel in {city}
    1. Grand Hotel - 5 star, $350/night, spa, gym, pool
    2. City Inn - 4 star, $1800/night, business center
    3. Budget Stay - 3 star, $350/night, free wifi
    """
    
agent = create_agent(
    model = "groq:openai/gpt-oss-120b",
    tools=[search_hostels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model = "groq:openai/gpt-oss-120b",
            trigger=("tokens",550),
            keep=("tokens",200)
        )
    ]
    
)

config = {"configurable":{"thread_id":"test-1"}}

def count_token(messages):
    total_chars = sum(len(str(m.content))for m in messages)
    return total_chars//4  # 4 chars = 1 token 
    

In [27]:
# run test 

cities = ['Paris',"London","Tokyo","New York","Dubai","Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages":[HumanMessage(content = f"find hotels in {city}")]},
        config=config #type:ignore
    )
    
    tokens = count_token(response["messages"])
    print(f"{city}: ~ {tokens}, tokens, {len(response["messages"])} messages")
    print(f"{(response["messages"])}")
    


Paris: ~ 1855, tokens, 4 messages
[HumanMessage(content='find hotels in Paris', additional_kwargs={}, response_metadata={}, id='a231fad7-2232-4091-a23b-a643e24cc317'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to find hotels in Paris. We have a function search_hostels which expects city string. We can call it.', 'tool_calls': [{'id': 'fc_23f71ee2-f2ea-4147-947c-09cf41bda8a1', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hostels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 130, 'total_tokens': 185, 'completion_time': 0.115438569, 'completion_tokens_details': {'reasoning_tokens': 27}, 'prompt_time': 0.00497602, 'prompt_tokens_details': None, 'queue_time': 0.317613215, 'total_time': 0.120414589}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_854fa9be4c', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='l

### Fraction 


In [28]:
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.tools import tool
from langchain.agents.middleware import SummarizationMiddleware

@tool 
def search_hotels(city:str) -> str:
    """search hotels."""
    return f"Hotels in {city} : grand Hotel $350, city Inn $180, Budget Stay $75"

# LOW fraction for testing!

agent = create_agent(
    model = "groq:openai/gpt-oss-120b",
    tools=[search_hostels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model = "groq:openai/gpt-oss-120b",
            trigger=("fraction",0.005),  # 0.5% == ~640 tokens
            keep=("fraction",0.002)  # 0.2% == ~256 tokens
        )
    ]
        
)

config={"configurable":{"thread_id":"test-1"}}

# token counter

def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# test 

cities = ["Paris","London","Tokyo","New York","Dubai","Singapore"]

for city in cities:
    response=agent.invoke({"messages":[HumanMessage(content=f"Hotels in {city}")]},
                            config=config #type:ignore
                            )
    
    tokens = count_tokens(response["messages"])
    fraction = tokens/131072 # context of gpt-oss-120b model
    print(f"{city} : ~{tokens} tokens ({fraction:.4%}), {len(response["messages"])} messages")
    print(response['messages'])

Paris : ~382 tokens (0.2914%), 4 messages
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='60e080b9-0325-470d-98ba-7d9e3e6f1ae4'), AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to answer user request: "Hotels in Paris". We can use the search_hostels function. Provide results. Probably need to call function.', 'tool_calls': [{'id': 'fc_8d212995-63a1-4194-a7b6-5c3b89a8a147', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hostels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 129, 'total_tokens': 188, 'completion_time': 0.12273231, 'completion_tokens_details': {'reasoning_tokens': 31}, 'prompt_time': 0.005338272, 'prompt_tokens_details': None, 'queue_time': 0.320875406, 'total_time': 0.128070582}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c800245357', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model


### Human In the Loop MiddleWare 

 --> Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

- High-stakes operations requiring human approval (e.g. database writes, financial transactions).

- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [30]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id:str) -> str:
    """Mock function to read an email by its ID"""
    return f"Email connect for ID : {email_id}"

def send_email_tool(recipient :str, subject: str, body:str) -> str:
    """Mock function to send an email. """
    return f"Email sent to {recipient} with subject '{subject}'"


In [38]:
agent = create_agent(
    model = "groq:openai/gpt-oss-120b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,
            }
        )
    ]
)

In [39]:
config = {"configurable":{"thread_id":"test-approve"}}

# Step-1: Request

result= agent.invoke(
    {"messages":[HumanMessage(content= "send email to john@test.com with subject 'hello' and body 'How are you?'")]},
    config=config #type:ignore
)

In [40]:
result

{'messages': [HumanMessage(content="send email to john@test.com with subject 'hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='aa8ccf66-9c65-427b-995d-1ff30fbe749c'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send email using send_email_tool. Provide parameters.', 'tool_calls': [{'id': 'fc_700963de-0207-44ef-a8f4-1c4a70c1cb75', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 174, 'total_tokens': 234, 'completion_time': 0.128599063, 'completion_tokens_details': {'reasoning_tokens': 14}, 'prompt_time': 0.008651335, 'prompt_tokens_details': None, 'queue_time': 0.212036893, 'total_time': 0.137250398}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_6dedd2be22', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': No

In [41]:
# step-2 : Approve 

from langgraph.types import Command

if "__interrupt__" in result:
    print("⏸️ paused! approving...")
    
    result = agent.invoke(
        Command(
            resume={
        
                "decisions":[
                    {"type":"approve"},
                ],
            },
        ),
        config= config #type:ignore
    )
    
    print(f"✅ Result : {result['messages'][-1].content}")

⏸️ paused! approving...
✅ Result : Your email has been sent to **john@test.com** with the subject **“hello”** and the body:

*How are you?*


In [42]:
result

{'messages': [HumanMessage(content="send email to john@test.com with subject 'hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='aa8ccf66-9c65-427b-995d-1ff30fbe749c'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send email using send_email_tool. Provide parameters.', 'tool_calls': [{'id': 'fc_700963de-0207-44ef-a8f4-1c4a70c1cb75', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 174, 'total_tokens': 234, 'completion_time': 0.128599063, 'completion_tokens_details': {'reasoning_tokens': 14}, 'prompt_time': 0.008651335, 'prompt_tokens_details': None, 'queue_time': 0.212036893, 'total_time': 0.137250398}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_6dedd2be22', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': No

In [48]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id:str) ->str:
    """ Mock function to read an email by its ID """
    
    return f"Email connect for ID : {email_id}"
    
def send_email_tool(recipient :str, subject: str, body:str) -> str:
    """Mock function to send an email. """
    return f"Email sent to {recipient} with subject '{subject}'"



agent = create_agent(
    model = "groq:openai/gpt-oss-120b",
    checkpointer=InMemorySaver(),
    tools=[send_email_tool,read_email_tool],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False
            }
        )
    ]
)

config = {"configurable":{"thread_id":"test-rejected"}}


In [49]:
# step-1: Requested 
result = agent.invoke(
    {
    'messages':[HumanMessage(content="send email to john@test.com with subject 'hello' and body 'How are you?'")]
    },
    config = config #type:ignore
)

In [50]:
result

{'messages': [HumanMessage(content="send email to john@test.com with subject 'hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='5ded0ff5-5f81-4b54-93b6-4344c1fa590a'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call send_email_tool with given parameters.', 'tool_calls': [{'id': 'fc_3f5f8160-a7ed-4b81-81a8-ef63f088b0e7', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 174, 'total_tokens': 232, 'completion_time': 0.121893474, 'completion_tokens_details': {'reasoning_tokens': 12}, 'prompt_time': 0.006935471, 'prompt_tokens_details': None, 'queue_time': 0.307030677, 'total_time': 0.128828945}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_f73454f048', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model

In [51]:
# step-2 : Reject 

from langgraph.types import Command

if "__interrupt__" in result:
    print("⏸️ paused! approving...")
    
    result = agent.invoke(
        Command(
            resume={
        
                "decisions":[
                    {"type":"reject"},
                ],
            },
        ),
        config= config #type:ignore
    )
    
    print(f"✅ Result : {result['messages'][-1].content}")

⏸️ paused! approving...
✅ Result : I’m sorry—I wasn’t able to send the email because the request to use the email‑sending tool was rejected. If you’d still like me to send that message, just let me know and I can try again.


In [52]:
result

{'messages': [HumanMessage(content="send email to john@test.com with subject 'hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='5ded0ff5-5f81-4b54-93b6-4344c1fa590a'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call send_email_tool with given parameters.', 'tool_calls': [{'id': 'fc_3f5f8160-a7ed-4b81-81a8-ef63f088b0e7', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 174, 'total_tokens': 232, 'completion_time': 0.121893474, 'completion_tokens_details': {'reasoning_tokens': 12}, 'prompt_time': 0.006935471, 'prompt_tokens_details': None, 'queue_time': 0.307030677, 'total_time': 0.128828945}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_f73454f048', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model

# Editing


In [62]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [63]:
config = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request (with wrong info)
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'")]},
    config=config #type:ignore
)

In [64]:

result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='c08d33ea-c688-4ab4-8a75-d186ad56c484'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "The user wants to send an email. We need to call send_email_tool with recipient wrong@email.com, subject 'Test', body 'Hello'.", 'tool_calls': [{'id': 'fc_b39dc495-23aa-4336-b8fb-8da6edfe7add', 'function': {'arguments': '{"body":"Hello","recipient":"wrong@email.com","subject":"Test"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 74, 'prompt_tokens': 172, 'total_tokens': 246, 'completion_time': 0.15500545, 'completion_tokens_details': {'reasoning_tokens': 30}, 'prompt_time': 0.008362106, 'prompt_tokens_details': None, 'queue_time': 0.401469751, 'total_time': 0.163367556}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c868cf1eaa', 'service_tier': 'on_deman

In [65]:
# Step 2: Edit and approve
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config #type:ignore
    )
    
    print(f"✏️ Result: {result['messages'][-1].content}")

⏸️ Paused! Editing...
✏️ Result: 


In [66]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='c08d33ea-c688-4ab4-8a75-d186ad56c484'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "The user wants to send an email. We need to call send_email_tool with recipient wrong@email.com, subject 'Test', body 'Hello'.", 'tool_calls': [{'id': 'fc_b39dc495-23aa-4336-b8fb-8da6edfe7add', 'function': {'arguments': '{"body":"Hello","recipient":"wrong@email.com","subject":"Test"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 74, 'prompt_tokens': 172, 'total_tokens': 246, 'completion_time': 0.15500545, 'completion_tokens_details': {'reasoning_tokens': 30}, 'prompt_time': 0.008362106, 'prompt_tokens_details': None, 'queue_time': 0.401469751, 'total_time': 0.163367556}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c868cf1eaa', 'service_tier': 'on_deman